# 面试问题：LLaVA 风格视觉语言 Projector、Token 拼接和训练怎样从零实现？

可直接复述的回答：视觉编码器先把图像切成 patch token，Projector 把视觉维度映射到语言模型 hidden size。视觉 token 在 prompt 中占据明确位置，并与文本 token 一起进入 causal decoder。训练标签只监督 assistant 回答，视觉和用户 token 不参与 loss。第一阶段常冻结视觉和语言骨干只训练 projector，第二阶段再按数据与显存选择解冻范围。图像尺寸、patch 顺序和 placeholder 数必须形成输入合同。评估要看 grounding、OCR、幻觉和错误样本，不只看训练 loss。真实 LLaVA 还依赖预训练视觉塔和大语言模型。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：包裹破损小图与输入预览

五张 4×4 灰度教学图模拟物流包裹照片：左上角高亮表示破损，均匀图表示完好。文本 prompt 相同，因此 text-only 无法区分；图像是可打印的小型脱敏图案。


In [1]:
import torch  # 使用 PyTorch 基础张量实现视觉 token 和模型。
from torch import nn  # 使用基础模块构造 projector 与注意力。
import torch.nn.functional as F  # 使用交叉熵训练多模态分类答案。
torch.manual_seed(20260729)  # 固定参数初始化和训练输出。
images09 = torch.tensor([[[1.0, 1.0, 0.1, 0.1], [1.0, 0.9, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1]], [[0.1, 0.1, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1]], [[0.9, 1.0, 0.2, 0.1], [1.0, 0.8, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1]], [[0.2, 0.2, 0.2, 0.2], [0.2, 0.2, 0.2, 0.2], [0.2, 0.2, 0.2, 0.2], [0.2, 0.2, 0.2, 0.2]], [[1.0, 0.8, 0.1, 0.1], [0.9, 1.0, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1], [0.1, 0.1, 0.1, 0.1]]], dtype=torch.float32)  # 构造五张包裹灰度图。
labels09 = torch.tensor([1, 0, 1, 0, 1], dtype=torch.long)  # 定义0完好、1破损标签。
prompt09 = "图片中的包裹是否破损"  # 所有样本使用相同文本问题。
print("教学实验输入：sample | prompt | ascii_image | label")  # 输出多模态输入表头。
for index09, image09 in enumerate(images09):  # 逐张展示可读图案。
    ascii09 = ["".join("#" if float(pixel09) > 0.6 else "." for pixel09 in row09) for row09 in image09]  # 将灰度图转换为字符预览。
    print(index09, prompt09, ascii09, "破损" if int(labels09[index09]) else "完好")  # 输出图像、问题和人工标签。


教学实验输入：sample | prompt | ascii_image | label
0 图片中的包裹是否破损 ['##..', '##..', '....', '....'] 破损
1 图片中的包裹是否破损 ['....', '....', '....', '....'] 完好
2 图片中的包裹是否破损 ['##..', '##..', '....', '....'] 破损
3 图片中的包裹是否破损 ['....', '....', '....', '....'] 完好
4 图片中的包裹是否破损 ['##..', '##..', '....', '....'] 破损


## 2. Baseline（基线）：只使用相同文本 Prompt

所有 prompt 完全相同，text-only 只能预测训练集多数类“破损”，无法解释每张图片差异。


In [2]:
majority_label09 = int(torch.mode(labels09).values)  # 获取 text-only 能利用的多数标签。
baseline_predictions09 = torch.full_like(labels09, majority_label09)  # 对所有相同文本返回同一答案。
baseline_accuracy09 = float((baseline_predictions09 == labels09).float().mean())  # 计算多数类准确率。
print("Text-only基线：sample | prediction | label")  # 输出基线逐样本表头。
for index09 in range(len(labels09)):  # 逐样本展示同一文本预测。
    print(index09, int(baseline_predictions09[index09]), int(labels09[index09]))  # 输出预测和人工标签。
print("Text-only准确率", round(baseline_accuracy09, 3))  # 展示缺失视觉证据的上限。


Text-only基线：sample | prediction | label
0 1 1
1 1 0
2 1 1
3 1 0
4 1 1
Text-only准确率 0.6


## 3. 核心实现：Patchify、Projector 与因果融合

每张 4×4 图切成四个 2×2 patch，Projector 映射到 8 维语言空间。一个文本 token 追加在视觉 token 后，手写单头 causal attention，最后用文本位置预测答案。


In [3]:
def patchify09(images09):  # 将批量图像切成固定顺序 patch token。
    patches09 = images09.unfold(1, 2, 2).unfold(2, 2, 2)  # 沿高宽生成不重叠2×2窗口。
    return patches09.contiguous().view(images09.shape[0], 4, 4)  # 按行优先得到四个四维 patch。
class TinyVLM09(nn.Module):  # 定义教学版视觉语言融合模型。
    def __init__(self, hidden09=8):  # 初始化 projector、文本 token 和注意力参数。
        super().__init__()  # 注册 PyTorch 模块状态。
        self.projector = nn.Linear(4, hidden09)  # 把四维 patch 映射到语言 hidden size。
        self.text_token = nn.Parameter(torch.randn(1, 1, hidden09) * 0.02)  # 表示固定用户问题 token。
        self.q = nn.Linear(hidden09, hidden09, bias=False)  # 定义 query 投影。
        self.k = nn.Linear(hidden09, hidden09, bias=False)  # 定义 key 投影。
        self.v = nn.Linear(hidden09, hidden09, bias=False)  # 定义 value 投影。
        self.head = nn.Linear(hidden09, 2)  # 从文本位置预测完好或破损。
    def forward(self, images09):  # 定义多模态前向计算。
        visual09 = self.projector(patchify09(images09))  # 生成四个视觉语言 token。
        text09 = self.text_token.expand(images09.shape[0], -1, -1)  # 为每张图复制同一问题 token。
        sequence09 = torch.cat([visual09, text09], dim=1)  # 按视觉后文本顺序拼接多模态 token。
        q09 = self.q(sequence09)  # 计算融合序列 query。
        k09 = self.k(sequence09)  # 计算融合序列 key。
        v09 = self.v(sequence09)  # 计算融合序列 value。
        scores09 = q09 @ k09.transpose(-2, -1) / (sequence09.shape[-1] ** 0.5)  # 计算缩放注意力得分。
        causal09 = torch.tril(torch.ones(sequence09.shape[1], sequence09.shape[1], dtype=torch.bool))  # 构造 causal mask。
        scores09 = scores09.masked_fill(~causal09, -1e9)  # 阻止视觉位置查看未来文本位置。
        hidden09 = torch.softmax(scores09, dim=-1) @ v09  # 聚合视觉和文本上下文。
        return self.head(hidden09[:, -1]), visual09, scores09  # 在最后文本位置输出答案 logits。
patches09 = patchify09(images09)  # 生成可检查视觉 patch。
model09 = TinyVLM09()  # 创建确定性教学多模态模型。
optimizer09 = torch.optim.Adam(model09.parameters(), lr=0.06)  # 使用基础优化器训练。
loss_trace09 = []  # 保存两阶段简化训练损失。
for _ in range(100):  # 运行少量全数据教学训练。
    optimizer09.zero_grad()  # 清空上一步梯度。
    logits09, visual09, scores09 = model09(images09)  # 执行视觉token拼接和因果融合。
    loss09 = F.cross_entropy(logits09, labels09)  # 只监督 assistant 答案标签。
    loss09.backward()  # 反向传播到 projector 和融合层。
    optimizer09.step()  # 更新教学模型参数。
    loss_trace09.append(float(loss09.detach()))  # 保存当前损失。
print("核心形状", {"images": tuple(images09.shape), "patches": tuple(patches09.shape), "visual_tokens": tuple(visual09.shape), "attention": tuple(scores09.shape)})  # 展示图像到语言token链路。
print("训练loss start -> end", round(loss_trace09[0], 4), "->", round(loss_trace09[-1], 4))  # 展示多模态训练过程。
print("首张图四个patch", patches09[0].tolist())  # 展示 patch 顺序和数值。


核心形状 {'images': (5, 4, 4), 'patches': (5, 4, 4), 'visual_tokens': (5, 4, 8), 'attention': (5, 5, 5)}
训练loss start -> end 0.7512 -> 0.0
首张图四个patch [[1.0, 1.0, 1.0, 0.8999999761581421], [0.10000000149011612, 0.10000000149011612, 0.10000000149011612, 0.10000000149011612], [0.10000000149011612, 0.10000000149011612, 0.10000000149011612, 0.10000000149011612], [0.10000000149011612, 0.10000000149011612, 0.10000000149011612, 0.10000000149011612]]


## 4. 结果表与结果解读

同一文本下，只有视觉 token 能区分完好和破损。教学模型在五张闭集图上拟合成功，仅证明 patch/projector/融合链路可运行，不能外推真实照片。


In [4]:
model09.eval()  # 切换到确定性评估模式。
with torch.no_grad():  # 关闭评估阶段梯度记录。
    final_logits09, _, _ = model09(images09)  # 计算五张图的最终 logits。
    predictions09 = final_logits09.argmax(dim=-1)  # 获取完好或破损预测。
core_accuracy09 = float((predictions09 == labels09).float().mean())  # 计算闭集教学准确率。
print("方法 | sample | prediction | label")  # 输出逐样本对照表头。
for index09 in range(len(labels09)):  # 逐图展示 text-only 与视觉模型结果。
    print("text_only", index09, int(baseline_predictions09[index09]), int(labels09[index09]))  # 输出文本基线结果。
    print("tiny_vlm", index09, int(predictions09[index09]), int(labels09[index09]))  # 输出多模态模型结果。
print("准确率对照", {"text_only": round(baseline_accuracy09, 3), "tiny_vlm": round(core_accuracy09, 3)})  # 汇总视觉贡献。
print("结果解读：相同问题无法提供区分信号，projector让图像patch进入语言hidden空间")  # 解释多模态收益来源。


方法 | sample | prediction | label
text_only 0 1 1
tiny_vlm 0 1 1
text_only 1 1 0
tiny_vlm 1 0 0
text_only 2 1 1
tiny_vlm 2 1 1
text_only 3 1 0
tiny_vlm 3 0 0
text_only 4 1 1
tiny_vlm 4 1 1
准确率对照 {'text_only': 0.6, 'tiny_vlm': 1.0}
结果解读：相同问题无法提供区分信号，projector让图像patch进入语言hidden空间


## 5. 失败案例与修正：Patch 顺序与 Placeholder 合同漂移

服务若把训练时 row-major patch 改成 column-major，位置语义会静默变化。修正是在输入 manifest 中绑定图像尺寸、patch size、顺序和视觉 token 数，加载前拒绝漂移。


In [5]:
training_manifest09 = {"height": 4, "width": 4, "patch": 2, "order": "row_major", "visual_tokens": 4}  # 定义训练时视觉输入合同。
serving_manifest09 = {"height": 4, "width": 4, "patch": 2, "order": "column_major", "visual_tokens": 4}  # 构造顺序漂移的服务配置。
failed_contract09 = training_manifest09 == serving_manifest09  # 演示只要顺序不同就不兼容。
fixed_serving09 = dict(training_manifest09)  # 使用发布制品恢复完全一致配置。
fixed_contract09 = training_manifest09 == fixed_serving09  # 检查修正后合同一致。
print("失败行为：训练/服务视觉合同", training_manifest09, serving_manifest09, "compatible=", failed_contract09)  # 展示静默顺序漂移。
print("修正行为：绑定manifest", fixed_serving09, "compatible=", fixed_contract09)  # 展示加载前固定配置。


失败行为：训练/服务视觉合同 {'height': 4, 'width': 4, 'patch': 2, 'order': 'row_major', 'visual_tokens': 4} {'height': 4, 'width': 4, 'patch': 2, 'order': 'column_major', 'visual_tokens': 4} compatible= False
修正行为：绑定manifest {'height': 4, 'width': 4, 'patch': 2, 'order': 'row_major', 'visual_tokens': 4} compatible= True


## 6. 生产边界与多模态制品

真实 LLaVA 使用预训练视觉塔和 LLM，并需要图像归一化、动态分辨率、视觉 placeholder、OCR/grounding 数据和两阶段冻结策略。还要评估图像幻觉和无图回退。


In [6]:
vlm_contract09 = {"vision_encoder": "frozen-vision-v5", "projector": "mlp-v2", "language_model": "lm-v8", "image_manifest": training_manifest09, "loss_mask": "assistant_only", "stage1": "projector", "stage2": "selected_unfreeze"}  # 定义视觉语言发布合同。
print("VLM 训练制品", vlm_contract09)  # 展示视觉塔、projector、LLM和输入配置版本。
print("生产替换点：预训练视觉塔、真实LLM、动态分辨率、OCR/grounding集、两阶段训练和幻觉评测")  # 说明微型闭集模型的边界。


VLM 训练制品 {'vision_encoder': 'frozen-vision-v5', 'projector': 'mlp-v2', 'language_model': 'lm-v8', 'image_manifest': {'height': 4, 'width': 4, 'patch': 2, 'order': 'row_major', 'visual_tokens': 4}, 'loss_mask': 'assistant_only', 'stage1': 'projector', 'stage2': 'selected_unfreeze'}
生产替换点：预训练视觉塔、真实LLM、动态分辨率、OCR/grounding集、两阶段训练和幻觉评测


## 7. 最小回归测试

断言保护案例规模、视觉 token、训练链路和输入合同。


In [7]:
assert len(images09) >= 5  # 保证多模态案例包含足够图像。
assert patches09.shape == (5, 4, 4)  # 保证每张图生成四个四维 patch token。
assert loss_trace09[-1] < loss_trace09[0]  # 保证视觉语言训练链路能够下降。
assert core_accuracy09 > baseline_accuracy09  # 保证视觉证据优于相同文本基线。
assert failed_contract09 is False and fixed_contract09 is True  # 保证 patch 顺序漂移被门禁修正。
print("最小回归测试通过：Patch、Projector、融合训练和视觉合同稳定")  # 显示多模态关键性质已验证。


最小回归测试通过：Patch、Projector、融合训练和视觉合同稳定
